# Grok-ML-gpu-smoke

Kaggle T4×2 GPU smoke: detect multi-GPU, run a tiny dual-device training loop, write `results.json`.

- **Domain**: ML
- **Task**: gpu-smoke
- **Naming**: `Grok-{领域}-{任务}`


In [ ]:
import json, os, platform, time, traceback
from pathlib import Path

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)
results = {
    "notebook": "Grok-ML-gpu-smoke",
    "started_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "host": platform.node(),
    "python": platform.python_version(),
    "ok": False,
    "errors": [],
}
print("cwd=", os.getcwd())
print("python=", platform.python_version())


In [ ]:
# --- GPU inventory ---
import torch

results["torch"] = torch.__version__
results["cuda_built"] = torch.version.cuda
results["cuda_available"] = bool(torch.cuda.is_available())
results["device_count"] = int(torch.cuda.device_count()) if torch.cuda.is_available() else 0

gpus = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        gpus.append({
            "index": i,
            "name": torch.cuda.get_device_name(i),
            "total_memory_gb": round(props.total_memory / (1024**3), 2),
            "multi_processor_count": props.multi_processor_count,
        })
results["gpus"] = gpus
print(json.dumps({"cuda_available": results["cuda_available"], "device_count": results["device_count"], "gpus": gpus}, indent=2))

assert torch.cuda.is_available(), "CUDA not available — expected T4x2 GPU session"
assert torch.cuda.device_count() >= 1, "Need at least 1 GPU"


In [ ]:
# --- Dual-GPU (or single) matmul + tiny train ---
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
n_gpus = torch.cuda.device_count()
device0 = torch.device("cuda:0")

class TinyMLP(nn.Module):
    def __init__(self, d=256, h=512, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, n_classes),
        )
    def forward(self, x):
        return self.net(x)

model = TinyMLP().to(device0)
if n_gpus >= 2:
    model = nn.DataParallel(model)
    results["parallel"] = "DataParallel"
else:
    results["parallel"] = "single"

opt = optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

t0 = time.time()
losses = []
for step in range(50):
    x = torch.randn(512, 256, device=device0)
    y = torch.randint(0, 10, (512,), device=device0)
    opt.zero_grad(set_to_none=True)
    logits = model(x)
    loss = loss_fn(logits, y)
    loss.backward()
    opt.step()
    losses.append(float(loss.item()))
    if step % 10 == 0:
        print(f"step={step:02d} loss={loss.item():.4f}")

# force sync + peak mem
if n_gpus >= 1:
    torch.cuda.synchronize()
    results["peak_mem_gb"] = {
        f"cuda:{i}": round(torch.cuda.max_memory_allocated(i) / (1024**3), 3)
        for i in range(n_gpus)
    }

elapsed = time.time() - t0
results["train_steps"] = 50
results["final_loss"] = losses[-1]
results["loss_curve_head"] = losses[:5]
results["loss_curve_tail"] = losses[-5:]
results["elapsed_sec"] = round(elapsed, 3)
print("final_loss=", results["final_loss"], "elapsed=", results["elapsed_sec"])
assert results["final_loss"] < losses[0], "loss did not decrease"
assert results["final_loss"] < 5.0, f"loss too high: {results['final_loss']}"


In [ ]:
# --- dual-device bandwidth microbench (if 2+ GPUs) ---
if torch.cuda.device_count() >= 2:
    a = torch.randn(2048, 2048, device="cuda:0")
    t0 = time.time()
    b = a.to("cuda:1")
    torch.cuda.synchronize()
    results["p2p_copy_sec"] = round(time.time() - t0, 4)
    c = torch.mm(b, b)
    torch.cuda.synchronize()
    results["remote_gemm_ok"] = True
    print("p2p_copy_sec=", results["p2p_copy_sec"], "remote_gemm ok")
else:
    results["p2p_copy_sec"] = None
    results["remote_gemm_ok"] = False
    print("single GPU — skip p2p microbench")


In [ ]:
# --- finalize ---
results["ok"] = True
results["finished_at"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
path = OUT / "results.json"
path.write_text(json.dumps(results, indent=2))
print(path.read_text())
print("SMOKE_OK")
